# Match Simbad Targets with LSSTCam Object Catalogue

For each target star (identified by `simbad_id`, `ra_deg`, `dec_deg`) read from
the input CSV, this notebook:

1. Locates the Butler tract/patch that covers the target coordinates.
2. Loads the corresponding `objectTable_tract` (or equivalent catalogue dataset
   auto-discovered from the registry).
3. Performs a nearest-neighbour sky cross-match using
   `astropy.coordinates.SkyCoord.match_to_catalog_sky`.
4. Retains matches within a configurable search radius.
5. Saves the merged table (targets + matched LSST object columns) to CSV/Parquet.

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-26
- **Last update:** 2026-06-26


## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from astropy.coordinates import SkyCoord
import astropy.units as u

from lsst.daf.butler import Butler
import lsst.geom as geom
from lsst.geom import SpherePoint, degrees

## Helper utilities

In [ ]:
def dataset_type_exists(butler, name):
    """Return True if *name* is a registered dataset type in *butler*."""
    try:
        butler.registry.getDatasetType(name)
        return True
    except KeyError:
        return False

## Configuration

**Edit only this cell** to point to the right Butler repository, collections,
input CSV, search radius, and output band.


In [ ]:
# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "MATCHOBJ_02"
DIR_DATA_IN = "data_DEEPCCUTOUTS_01_in"  # reuse the same input directory as NB01
DIR_DATA_OUT = f"data_{NB_TAG}_out"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA_OUT, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data input : {os.path.abspath(DIR_DATA_IN)}")
print(f"Data output: {os.path.abspath(DIR_DATA_OUT)}")
print(f"Figs       : {os.path.abspath(DIR_FIGS)}")

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure to PDF and PNG inside DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

In [ ]:
# ── Butler ────────────────────────────────────────────────────────────────────
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

instrument = "LSSTCam"
skymapName = "lsst_cells_v2"

# ── Input targets ─────────────────────────────────────────────────────────────
target_file = "summary_visit_counts_per_star_V17-21_r2.0deg.csv"

# ── Cross-match search radius ─────────────────────────────────────────────────
MATCH_RADIUS_ARCSEC = 1.0  # maximum separation for a valid match [arcsec]

# ── Band used to retrieve the object table (if band-split tables are used) ────
BANDSEL = "r"

print("Butler configuration done.")

## Load targets

In [ ]:
df_targets = pd.read_csv(os.path.join(DIR_DATA_IN, target_file))
print(f"Loaded {len(df_targets)} targets from {target_file}")
display(df_targets.head())

## Initialise the Butler

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
print(f"Butler initialised | repo: {repo}")

## Auto-discover the object-table dataset type

The catalogue dataset name changed across pipeline versions:

| Pipeline era | Dataset type name          |
|:-------------|:---------------------------|
| Gen2 / HSC   | `deepCoadd_obj`            |
| DP1          | `objectTable_tract`        |
| DP2+         | `object_table_tract`       |

We probe the registry so the notebook is collection-agnostic.


In [ ]:
# Prioritised list of candidate object-table dataset type names
OBJECT_TABLE_CANDIDATES = [
    "object",
    "objectTable_tract",  # DP1 / recent Science Pipelines
    "object_table_tract",  # DP2+
    "deepCoadd_obj",  # Gen2 / older collections
    "sourceTable_visit",  # visit-level source table (fallback)
]

# List all object-table-related types actually in the registry
all_obj_types = [
    d.name for d in registry.queryDatasetTypes() if "object" in d.name.lower() or "table" in d.name.lower()
]
print("Object-table-related dataset types in registry:")
for t in sorted(all_obj_types):
    print(f"  {t}")

# Pick the first candidate that is registered
OBJ_DATASET = None
for name in OBJECT_TABLE_CANDIDATES:
    if dataset_type_exists(butler, name):
        OBJ_DATASET = name
        print(f"\n✔ Selected object-table dataset type: '{OBJ_DATASET}'")
        break

if OBJ_DATASET is None:
    raise RuntimeError(
        "No recognised object-table dataset type found in this Butler collection. "
        f"Candidate types seen: {all_obj_types}"
    )

## Inspect the schema of the object table (first available tract)

We load a single object table to check which columns are available before
the main loop.

In [ ]:
# Use the coordinates of the first target to locate a representative tract
first_row = df_targets.iloc[0]
probe_point = SpherePoint(first_row["ra_deg"], first_row["dec_deg"], degrees)
probe_tract = skymap.findTract(probe_point)
probe_tract_id = probe_tract.getId()

probe_dataId = {"tract": probe_tract_id, "skymap": skymapName}

print(f"Probing schema with tract {probe_tract_id} …")
df_probe = butler.get(OBJ_DATASET, dataId=probe_dataId)

# Convert to pandas if the Butler returns an lsst.afw table or an Arrow table
if not isinstance(df_probe, pd.DataFrame):
    try:
        df_probe = df_probe.to_pandas()
    except AttributeError:
        df_probe = pd.DataFrame(df_probe)

print(f"  Shape: {df_probe.shape}")
print(f"  Columns ({len(df_probe.columns)}):")
for col in sorted(df_probe.columns):
    print(f"    {col}")

In [ ]:
# Identify the RA/Dec and objectId column names from the schema
# They can vary between pipeline versions.


def find_col(df, candidates):
    """Return the first column name from *candidates* that exists in *df*."""
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of {candidates} found in table columns: {list(df.columns[:20])}")


RA_COL = find_col(df_probe, ["coord_ra", "ra", "RA", "ra_deg"])
DEC_COL = find_col(df_probe, ["coord_dec", "dec", "DEC", "dec_deg"])
ID_COL = find_col(df_probe, ["objectId", "object_id", "id", "sourceId"])

print(f"RA column  : {RA_COL}")
print(f"Dec column : {DEC_COL}")
print(f"ID column  : {ID_COL}")

## Main cross-match loop

For every target we:
1. Find the tract that contains the target.
2. Load `objectTable_tract` for that tract (cached: each tract is loaded only once).
3. Cross-match with `SkyCoord.match_to_catalog_sky`.
4. Keep the match only if separation ≤ `MATCH_RADIUS_ARCSEC`.


In [ ]:
# Cache: avoid reloading the same tract table multiple times
tract_cache: dict[int, pd.DataFrame] = {}


def get_object_table(tract_id: int) -> pd.DataFrame:
    """Load (and cache) the object table for *tract_id*."""
    if tract_id in tract_cache:
        return tract_cache[tract_id]

    dataId = {"tract": tract_id, "skymap": skymapName}
    print(f"  Loading object table for tract {tract_id} …", end=" ", flush=True)
    try:
        df = butler.get(OBJ_DATASET, dataId=dataId)
        if not isinstance(df, pd.DataFrame):
            try:
                df = df.to_pandas()
            except AttributeError:
                df = pd.DataFrame(df)
        print(f"{len(df):,} objects")
        tract_cache[tract_id] = df
        return df
    except Exception as exc:
        print(f"FAILED ({exc})")
        return None


print("Tract cache initialised.")

In [ ]:
results = []  # list of dicts; one per target

for idx, target in df_targets.iterrows():
    simbad_id = target["simbad_id"]
    ra_t = target["ra_deg"]
    dec_t = target["dec_deg"]

    # ── 1.  Find tract ────────────────────────────────────────────────────────
    point = SpherePoint(ra_t, dec_t, degrees)
    tract_info = skymap.findTract(point)
    tract_id = tract_info.getId()

    print(f"[{idx:3d}] {simbad_id:40s}  ra={ra_t:.5f}  dec={dec_t:+.5f}  tract={tract_id}")

    # ── 2.  Load object table (cached) ────────────────────────────────────────
    df_obj = get_object_table(tract_id)

    if df_obj is None or len(df_obj) == 0:
        print(f"       *** no object table available for tract {tract_id} — skipping")
        row = target.to_dict()
        row.update(
            {
                "lsst_objectId": None,
                "lsst_ra": None,
                "lsst_dec": None,
                "separation_arcsec": None,
                "match_status": "no_table",
                "lsst_tract": tract_id,
            }
        )
        results.append(row)
        continue

    # ── 3.  Build SkyCoord catalogue ──────────────────────────────────────────
    # RA/Dec in the object table may be in degrees or radians depending on
    # the pipeline version.  Values ≤ 2π suggest radians.
    ra_obj = df_obj[RA_COL].values
    dec_obj = df_obj[DEC_COL].values

    if ra_obj.max() <= 2 * np.pi + 0.1:  # heuristic: radians
        unit_sky = u.rad
    else:  # degrees
        unit_sky = u.deg

    cat_sky = SkyCoord(ra=ra_obj * unit_sky, dec=dec_obj * unit_sky)
    tgt_sky = SkyCoord(ra=ra_t * u.deg, dec=dec_t * u.deg)

    # ── 4.  Nearest-neighbour match ───────────────────────────────────────────
    best_idx, sep2d, _ = tgt_sky.match_to_catalog_sky(cat_sky)
    sep_arcsec = sep2d.to(u.arcsec).value

    matched_obj = df_obj.iloc[best_idx]

    row = target.to_dict()
    row["lsst_tract"] = tract_id

    if sep_arcsec <= MATCH_RADIUS_ARCSEC:
        lsst_ra = matched_obj[RA_COL]
        lsst_dec = matched_obj[DEC_COL]
        if unit_sky == u.rad:
            lsst_ra = np.degrees(lsst_ra)
            lsst_dec = np.degrees(lsst_dec)

        row.update(
            {
                "lsst_objectId": int(matched_obj[ID_COL]),
                "lsst_ra": lsst_ra,
                "lsst_dec": lsst_dec,
                "separation_arcsec": sep_arcsec,
                "match_status": "matched",
            }
        )
        print(f"       ✔ matched  objectId={matched_obj[ID_COL]}  sep={sep_arcsec:.3f}")
    else:
        row.update(
            {
                "lsst_objectId": None,
                "lsst_ra": None,
                "lsst_dec": None,
                "separation_arcsec": sep_arcsec,
                "match_status": "no_match",
            }
        )
        print(f"       ✗ no match  (closest sep={sep_arcsec:.3f} arcsec > {MATCH_RADIUS_ARCSEC} arcsec)")

    results.append(row)

print(f"\nDone: {len(results)} targets processed.")

## Results summary

In [ ]:
df_results = pd.DataFrame(results)

n_total = len(df_results)
n_matched = (df_results["match_status"] == "matched").sum()
n_nomatch = (df_results["match_status"] == "no_match").sum()
n_notable = (df_results["match_status"] == "no_table").sum()

print(f"Total targets : {n_total}")
print(f"  Matched     : {n_matched}  ({100*n_matched/n_total:.1f}%)")
print(f"  No match    : {n_nomatch}  (sep > {MATCH_RADIUS_ARCSEC} arcsec)")
print(f"  No table    : {n_notable}  (Butler error)")

display(df_results)

## Diagnostic plots

In [ ]:
df_matched = df_results[df_results["match_status"] == "matched"].copy()

# ── Distribution of separations ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df_matched["separation_arcsec"], bins=20, edgecolor="k", linewidth=0.5)
ax.axvline(MATCH_RADIUS_ARCSEC, color="red", linestyle="--", label=f'search radius = {MATCH_RADIUS_ARCSEC}"')
ax.set_xlabel("Separation (arcsec)")
ax.set_ylabel("Number of targets")
ax.set_title("Cross-match separation — Simbad targets vs LSST objects")
ax.legend()
plt.tight_layout()
savefig("crossmatch_separation_histogram")
plt.show()

In [ ]:
# ── RA/Dec offset (target − LSST object) ─────────────────────────────────────
cos_dec = np.cos(np.radians(df_matched["dec_deg"].values))
d_ra = (df_matched["ra_deg"].values - df_matched["lsst_ra"].values) * cos_dec * 3600  # arcsec
d_dec = (df_matched["dec_deg"].values - df_matched["lsst_dec"].values) * 3600  # arcsec

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(d_ra, d_dec, s=20, alpha=0.7)
circle = plt.Circle(
    (0, 0), MATCH_RADIUS_ARCSEC, color="red", fill=False, linestyle="--", label=f'r = {MATCH_RADIUS_ARCSEC}"'
)
ax.add_patch(circle)
ax.axhline(0, color="k", linewidth=0.5)
ax.axvline(0, color="k", linewidth=0.5)
ax.set_xlabel(r"$\Delta\,\alpha\cos\delta$ (arcsec)")
ax.set_ylabel(r"$\Delta\,\delta$ (arcsec)")
ax.set_title("Positional offsets: Simbad − LSST")
ax.set_aspect("equal")
ax.legend()
plt.tight_layout()
savefig("crossmatch_radec_offsets")
plt.show()

In [ ]:
# ── Summary pie chart ─────────────────────────────────────────────────────────
counts = df_results["match_status"].value_counts()
fig, ax = plt.subplots(figsize=(4, 4))
ax.pie(counts.values, labels=counts.index, autopct="%1.0f%%", startangle=90)
ax.set_title(f'Match status (r < {MATCH_RADIUS_ARCSEC}" )')
plt.tight_layout()
savefig("crossmatch_status_pie")
plt.show()

## Save results

In [ ]:
out_stem = os.path.join(DIR_DATA_OUT, "targets_matched_lsst_objects")

df_results.to_csv(out_stem + ".csv", index=False)
df_results.to_parquet(out_stem + ".parquet", index=False)

print(f"Full results  → {out_stem}.csv  /  .parquet")
print(f"  {len(df_results)} rows total, {n_matched} matched")

# Save the matched-only subset for downstream notebooks
out_matched = os.path.join(DIR_DATA_OUT, "targets_matched_only")
df_matched.to_csv(out_matched + ".csv", index=False)
df_matched.to_parquet(out_matched + ".parquet", index=False)
print(f"Matched only  → {out_matched}.csv  /  .parquet")

## Additional columns from the LSST object table

The cell below enriches the matched table with extra LSST photometric columns
(PSF flux, extended-ness classifier, …) that may be useful in downstream
notebooks.


In [ ]:
# Columns to transfer from the LSST object table to the matched result
# (filter to those actually present in the schema discovered above)
EXTRA_COLS_WANTED = [
    # Photometry
    "r_psfFlux",
    "r_psfFluxErr",
    "g_psfFlux",
    "g_psfFluxErr",
    "i_psfFlux",
    "i_psfFluxErr",
    # Morphology / star-galaxy separator
    "r_extendedness",
    "r_sersicIndex",
    # Shape
    "r_ixx",
    "r_iyy",
    "r_ixy",
    # Flags
    "detect_isPrimary",
    "r_pixelFlags_saturated",
    "r_pixelFlags_cr",
]

# Restrict to columns that actually exist in the probe table
extra_cols = [c for c in EXTRA_COLS_WANTED if c in df_probe.columns]
print(f"Extra columns found in schema ({len(extra_cols)}/{len(EXTRA_COLS_WANTED)}):")
print("  ", extra_cols)

if extra_cols:
    # Build a look-up DataFrame keyed by objectId
    enriched_rows = []

    for tract_id, df_obj in tract_cache.items():
        sub = df_results[
            (df_results["lsst_tract"] == tract_id) & (df_results["match_status"] == "matched")
        ].copy()
        if len(sub) == 0:
            continue

        obj_ids = sub["lsst_objectId"].astype(int).values
        cols_to_get = [ID_COL] + extra_cols
        cols_present = [c for c in cols_to_get if c in df_obj.columns]

        obj_sub = df_obj.set_index(ID_COL)[extra_cols]
        sub = sub.set_index("lsst_objectId").join(obj_sub, how="left").reset_index()
        enriched_rows.append(sub)

    if enriched_rows:
        df_enriched = pd.concat(enriched_rows, ignore_index=True)

        out_enriched = os.path.join(DIR_DATA_OUT, "targets_matched_enriched")
        df_enriched.to_csv(out_enriched + ".csv", index=False)
        df_enriched.to_parquet(out_enriched + ".parquet", index=False)
        print(f"Enriched table → {out_enriched}.csv  /  .parquet")
        display(df_enriched.head())
    else:
        print("No enriched rows to save.")
else:
    print("None of the requested extra columns were found — skipping enrichment.")